# Custom Block Compare (Baseline vs Custom)

Ноутбук для низкоуровневых экспериментов:
- базовая модель `HomeForCausalLM`
- кастомная модель с заменой FFN блока
- сравнение обучения в одном цикле с `tqdm`
- график loss для обеих архитектур

In [ ]:
from pathlib import Path
import random

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import plotly.graph_objects as go

from transformers import AutoTokenizer, DataCollatorForLanguageModeling

from homellm.models.home_model import HomeConfig, HomeForCausalLM
from homellm.training.pretrain import StreamingTextDataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
# ===== Data =====
DATA_PATH = '/app/datasets/fineweb-2_train.jsonl'  # поменяй при необходимости
ensure_pretrain_dataset(DATA_PATH)  # скачает с HF, если файла нет
TOKENIZER_ID = 'gpt2'
SEQ_LEN = 1024
BATCH_SIZE = 2
MAX_STEPS = 150

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<|pad|>'})

dataset = StreamingTextDataset(DATA_PATH, tokenizer, seq_len=SEQ_LEN)
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, collate_fn=collator, num_workers=0)

print('vocab:', len(tokenizer))

In [ ]:
# ===== Base architecture config =====
cfg = HomeConfig(
    vocab_size=len(tokenizer),
    hidden_size=512,
    num_hidden_layers=8,
    num_attention_heads=8,
    max_position_embeddings=SEQ_LEN,
    use_sdpa=True,
    use_liger=True,
)

In [ ]:
# ===== Define custom FFN block =====
class CustomFFN(nn.Module):
    """
    Пример кастомного FFN вместо стандартного SwiGLU.
    Можно свободно менять под свои эксперименты.
    """
    def __init__(self, config):
        super().__init__()
        hidden = config.intermediate_size
        self.fc1 = nn.Linear(config.hidden_size, hidden, bias=False)
        self.fc2 = nn.Linear(hidden, config.hidden_size, bias=False)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.fc2(self.act(self.fc1(x))))

In [ ]:
# ===== Build two models: baseline and custom =====
baseline = HomeForCausalLM(cfg)
baseline.resize_token_embeddings(len(tokenizer))

custom = HomeForCausalLM(cfg)
custom.resize_token_embeddings(len(tokenizer))

# Заменяем FFN во всех блоках custom-модели
for block in custom.model.layers:
    block.mlp = CustomFFN(cfg)

baseline = baseline.to(DEVICE)
custom = custom.to(DEVICE)

print('baseline params:', sum(p.numel() for p in baseline.parameters()))
print('custom params:  ', sum(p.numel() for p in custom.parameters()))

In [ ]:
# ===== Optimizers =====
LR = 3e-4
GRAD_ACCUM = 4

opt_base = torch.optim.AdamW(baseline.parameters(), lr=LR, weight_decay=0.1)
opt_custom = torch.optim.AdamW(custom.parameters(), lr=LR, weight_decay=0.1)

In [ ]:
# ===== Joint training loop with tqdm =====
baseline.train()
custom.train()

opt_base.zero_grad(set_to_none=True)
opt_custom.zero_grad(set_to_none=True)

base_loss_hist = []
custom_loss_hist = []

pbar = tqdm(total=MAX_STEPS, desc='baseline vs custom')
step = 0

for batch in loader:
    input_ids = batch['input_ids'].to(DEVICE)
    labels = batch['labels'].to(DEVICE)
    attention_mask = batch.get('attention_mask')
    if attention_mask is not None:
        attention_mask = attention_mask.to(DEVICE)

    with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
        out_b = baseline(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        out_c = custom(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss_b = out_b.loss / GRAD_ACCUM
        loss_c = out_c.loss / GRAD_ACCUM

    loss_b.backward()
    loss_c.backward()

    if (step + 1) % GRAD_ACCUM == 0:
        torch.nn.utils.clip_grad_norm_(baseline.parameters(), 1.0)
        torch.nn.utils.clip_grad_norm_(custom.parameters(), 1.0)
        opt_base.step()
        opt_custom.step()
        opt_base.zero_grad(set_to_none=True)
        opt_custom.zero_grad(set_to_none=True)

    lb = float(loss_b.detach().item() * GRAD_ACCUM)
    lc = float(loss_c.detach().item() * GRAD_ACCUM)
    base_loss_hist.append(lb)
    custom_loss_hist.append(lc)

    pbar.set_postfix({'base': f'{lb:.4f}', 'custom': f'{lc:.4f}'})
    pbar.update(1)
    step += 1

    if step >= MAX_STEPS:
        break

pbar.close()
print('final base loss:', base_loss_hist[-1])
print('final custom loss:', custom_loss_hist[-1])

In [ ]:
# ===== Loss curves side-by-side =====
fig = go.Figure()
fig.add_trace(go.Scatter(y=base_loss_hist, mode='lines', name='baseline (SwiGLU)'))
fig.add_trace(go.Scatter(y=custom_loss_hist, mode='lines', name='custom (GELU FFN)'))
fig.update_layout(
    title='Training Loss: Baseline vs Custom Block',
    xaxis_title='step',
    yaxis_title='loss',
    template='plotly_dark',
    height=520,
)
fig.show()

In [ ]:
# Optional: save both variants
OUT_DIR = Path('/app/out/playground_compare')
OUT_DIR.mkdir(parents=True, exist_ok=True)

baseline.save_pretrained(OUT_DIR / 'baseline')
custom.save_pretrained(OUT_DIR / 'custom_ffn')
tokenizer.save_pretrained(OUT_DIR / 'baseline')
tokenizer.save_pretrained(OUT_DIR / 'custom_ffn')

print('saved to', OUT_DIR)